# LangChain 单 Agent Demo

使用 **DeepSeek V4 Pro**（`deepseek-v4-pro`）+ Tavily 搜索 + WeatherStack 天气工具，查询**上海当前天气**。

本 notebook 适配 **LangChain 1.x**（`create_agent`），不再依赖已移除的 `langchain.hub` / `create_react_agent` / `AgentExecutor`。

`.env` 需要：`DEEPSEEK_API_KEY`、`TAVILY_API_KEY`、`WEATHERSTACK_API_KEY`（可选 `DEEPSEEK_BASE_URL`）。

## 逻辑总览（彩色 Mermaid）

下图展示 Agent 的执行闭环：LLM 决策 → 需要时调用工具 → 把工具结果写回消息 → 再决策，直到给出最终回答。

```mermaid
%%{init: {
  "theme": "base",
  "themeVariables": {
    "fontSize": "14px",
    "fontFamily": "ui-sans-serif, system-ui",
    "primaryTextColor": "#0f172a",
    "lineColor": "#64748b"
  }
}}%%
flowchart TD
    U([用户问题]) --> AG[create_agent 图]
    AG --> LLM["deepseek-v4-pro\n决定是否调工具"]

    LLM --> DEC{需要工具?}
    DEC -->|否| FA[最终回答]
    DEC -->|是| ACT[选择 Tool Call]

    ACT --> T1[tavily_search]
    ACT --> T2[get_weather_data]

    T1 --> OBS[Tool Message]
    T2 --> OBS
    OBS --> MSG[写回 messages]
    MSG --> LLM

    FA --> OUT([返回 messages])

    classDef user fill:#14b8a6,stroke:#0f766e,stroke-width:2px,color:#042f2e
    classDef exec fill:#38bdf8,stroke:#0284c7,stroke-width:2px,color:#0c4a6e
    classDef model fill:#a78bfa,stroke:#7c3aed,stroke-width:2px,color:#2e1065
    classDef decide fill:#fbbf24,stroke:#d97706,stroke-width:2px,color:#422006
    classDef action fill:#fb7185,stroke:#e11d48,stroke-width:2px,color:#4c0519
    classDef tool fill:#34d399,stroke:#059669,stroke-width:2px,color:#064e3b
    classDef obs fill:#f472b6,stroke:#db2777,stroke-width:2px,color:#500724
    classDef mem fill:#94a3b8,stroke:#475569,stroke-width:2px,color:#0f172a
    classDef answer fill:#4ade80,stroke:#16a34a,stroke-width:2px,color:#14532d

    class U,OUT user
    class AG exec
    class LLM model
    class DEC decide
    class ACT action
    class T1,T2 tool
    class OBS obs
    class MSG mem
    class FA answer
```

**本例路径：** 用户询问「上海天气」→ Agent 调用 `get_weather_data("Shanghai")` → 观测温度/天气/湿度 → 汇总最终回答。

## 1. 导入依赖

引入环境变量加载、DeepSeek 兼容的 ChatOpenAI、Tavily 搜索工具、自定义 `@tool` 装饰器，以及 HTTP 请求库。

> LangChain 1.x 中已无 `from langchain import hub`，本 notebook 改为本地 `system_prompt`，无需再拉 Hub 模板。

In [1]:
import os
import certifi
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain.tools import tool
import requests

C:\Users\86137\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\triton\windows_utils.py:372: UserWarning: Failed to find CUDA.
  warnings.warn("Failed to find CUDA.")


## 2. 导入 Agent 工厂

LangChain 1.x 使用 `create_agent`：传入 model + tools，返回可 `invoke` 的 Agent 图（内部自动做工具循环）。

In [2]:
from langchain.agents import create_agent

## 3. 加载环境变量

设置 SSL 证书路径并读取 `.env` 中的 DeepSeek、Tavily、WeatherStack API Key。

In [16]:
# ==========================================
# LOAD ENV VARIABLES
# 统一读取最上层：人工智能面试题/.env（从 cwd 向上查找）
# ==========================================
from dotenv import find_dotenv

# os.environ["SSL_CERT_FILE"] = certifi.where()
env_path = find_dotenv(usecwd=True)
load_dotenv(env_path, override=True)
print("loaded .env from:", env_path or "(not found)")

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL", "https://api.deepseek.com")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")

loaded .env from: d:\workspace\doc\面试狂魔\人工智能面试题\AI_coding_interview\.env


## 4. 创建 Tavily 搜索工具

实例化联网搜索工具，最多返回 2 条结果，供 Agent 在需要补充信息时调用。

In [5]:
search_tool = TavilySearchResults(max_results=2)

C:\Users\86137\AppData\Local\Temp\ipykernel_47128\919418145.py:1: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  search_tool = TavilySearchResults(max_results=2)


## 5. 定义天气工具

用 `@tool` 把 WeatherStack API 封装成 Agent 可调用的函数：传入城市名，返回温度、天气描述和湿度。

In [6]:
@tool
def get_weather_data(city: str) -> str:
    """
    Fetch current weather information for a city.
    """

    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )

    response = requests.get(url)

    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )

## 6. 单独测试搜索工具

先不走 Agent，直接调用 Tavily，确认搜索工具能正常返回上海相关结果。

In [7]:
result = search_tool.invoke("Shanghai current weather")
result

[{'title': 'Shanghai, Shanghai Municipality, China 14 day weather forecast',
  'url': 'https://www.timeanddate.com/weather/china/shanghai/ext',
  'content': '| Wed Aug 19 | Image 18: Partly cloudy. | 109 / 82°F | Partly cloudy. | 152°F | 9 mph | ↑ | 53% | 6% | 0.00" | 3(Moderate) | 5:21 am | 6:33 pm |\n|  Updated Wednesday, August 5, 2026 5:24:06 am Shanghai time - Weather by CustomWeather, © 2026 |',
  'score': 0.8282873},
 {'title': 'Shanghai Weather in August 2026: Very Hot, Typhoon Season',
  'url': 'https://www.travelchinaguide.com/cityguides/shanghai/weather-august.htm',
  'content': '| 24 | 25.0°C / 77.0°F | 22.7°C / 72.9°F | 05:26 / 18:27 | 84 |\n| 25 | 24.4°C / 75.9°F | 23.4°C / 74.1°F | 05:26 / 18:26 | 91 |\n| 26 | 27.8°C / 82.0°F | 24.3°C / 75.7°F | 05:27 / 18:25 | 85 |\n| 27 | 29.3°C / 84.7°F | 24.7°C / 76.5°F | 05:27 / 18:23 | 83 |\n| 28 | 31.1°C / 88.0°F | 26.5°C / 79.7°F | 05:28 / 18:22 | 80 |\n| 29 | 30.0°C / 86.0°F | 26.1°C / 79.0°F | 05:29 / 18:21 | 84 |\n| 30 | 29.2°

## 7. 初始化 LLM

通过 OpenAI 兼容接口连接 DeepSeek V4 Pro，温度设为 0，便于 Agent 稳定选择工具。

In [8]:
# ==========================================
# LLM — DeepSeek V4 Pro（OpenAI 兼容接口）
# ==========================================

llm = ChatOpenAI(
    model="deepseek-v4-pro",
    temperature=0,
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
)

## 8. 单独测试 LLM

向模型发一条简单请求，确认 API Key 与 base_url 配置正确。

In [9]:
response = llm.invoke("用一句话介绍上海今天可能的天气特点")
response

AIMessage(content='今天（4月28日）上海天气主打“春如四季”，早晨清凉开篇，午后阳光助力气温回升，但昼夜温差大，仿佛一日之间穿越两季。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 159, 'prompt_tokens': 12, 'total_tokens': 171, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 120, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 12}, 'model_provider': 'openai', 'model_name': 'deepseek-v4-pro', 'system_fingerprint': 'fp_9954b31ca7_prod0820_fp8_kvcache_20260402', 'id': 'b3e94337-be4d-45b2-a407-7786de06595a', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019fd02c-d8e0-7ba3-8cae-af333432b40e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 12, 'output_tokens': 159, 'total_tokens': 171, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 120}})

## 9. 组装工具列表

把搜索工具和天气工具放进同一个 list，后续一并交给 Agent。

In [10]:
# ==========================================
# TOOLS
# ==========================================

tools = [search_tool, get_weather_data]

## 10. 创建 Agent

用 `create_agent` 组合 LLM、工具和系统提示词。返回的是可直接 `invoke` 的图，无需再包一层 AgentExecutor。

In [11]:
# ==========================================
# CREATE AGENT（LangChain 1.x）
# ==========================================

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=(
        "You are a helpful assistant. "
        "Use tools when needed to answer questions about weather and current information. "
        "Prefer get_weather_data for weather queries."
    ),
)

## 11. 运行 Agent：查询上海天气

向 Agent 提问「上海当前天气」。输入格式是 `messages` 列表。预期路径：调用 `get_weather_data`（城市为 Shanghai），再输出最终回答。

In [12]:
# ==========================================
# RUN — 查询上海天气
# ==========================================

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": (
                "What is the current weather in Shanghai? "
                "Please report temperature, weather description, and humidity."
            ),
        }
    ]
})

## 12. 打印最终答案

从返回的 `messages` 中取最后一条 AI 消息内容，即为 Agent 汇总后的上海天气说明。

In [13]:
final_message = result["messages"][-1]
print(final_message.content)

Here's the current weather for **Shanghai**, based on the latest available report:

| Detail | Value |
|---|---|
| **Temperature** | 88°F (31°C) — today's high expected around 96°F (36°C) |
| **Weather Description** | Mostly clear/partly cloudy (precipitation chance near 0%) |
| **Humidity** | **75%** |
| **Dew Point** | 79°F (26°C) |
| **Pressure** | 29.77 inHg |
| **Wind** | Light northeast wind |

**Summary:** Shanghai is hot and humid right now, with temperatures climbing into the mid-90s Fahrenheit (mid-30s Celsius) later in the day. The humidity at 75% makes it feel even warmer. Rain is not expected today, so it should remain dry with partly to mostly clear skies.

> ⚠️ **Note:** The weather data tool returned an error, so this information was sourced from third-party weather reports and may not be perfectly real-time. For the most up-to-date conditions, you may want to check a dedicated weather service like timeanddate.com or your local weather provider.


In [15]:
from pprint import pprint

# 逐条打印消息（最易读）；如需看完整 dict 结构，取消下一行注释
for msg in result["messages"]:
    msg.pretty_print()

# pprint(result, width=100, depth=4)

================================ Human Message =================================

What is the current weather in Shanghai? Please report temperature, weather description, and humidity.
================================== Ai Message ==================================
Tool Calls:
  get_weather_data (call_00_gBw3tDuILegGB3JbxoVo8875)
 Call ID: call_00_gBw3tDuILegGB3JbxoVo8875
  Args:
    city: Shanghai
================================= Tool Message =================================
Name: get_weather_data

Could not fetch weather data for Shanghai
================================== Ai Message ==================================

It seems the direct weather fetch didn't return data for Shanghai. Let me try looking it up another way.
Tool Calls:
  tavily_search_results_json (call_00_EG4EfYBJTIE8YUEVrLzR6621)
 Call ID: call_00_EG4EfYBJTIE8YUEVrLzR6621
  Args:
    query: current weather Shanghai temperature humidity
================================= Tool Message =================================

## 13. 本次运行实际 Workflow（根据日志）

这次运行走的是典型 **ReAct**（Reason + Act）循环：

> **Thought → Action → Observation →（不够则回到 Thought）…**  
> 直到 Observation 足够，才 **出环** 给出 Final Answer。

本例在同一条环上转了 **3 圈**（天气失败 → 搜索不够实时 → refine 成功）后才出环。

### ReAct 循环图（本例转 3 圈）

```mermaid
%%{init: {
  "theme": "base",
  "themeVariables": {
    "fontSize": "14px",
    "fontFamily": "ui-sans-serif, system-ui",
    "primaryTextColor": "#0f172a",
    "lineColor": "#64748b"
  }
}}%%
flowchart TD
    Q(["Question<br/>上海当前天气？"]) --> TH

    TH["Thought<br/>LLM 推理 / 决定下一步"]
    TH --> DEC{观测够了?}
    DEC -->|否 · 继续环<br/>↻1 ↻2 ↻3| ACT
    DEC -->|是 · 出环| FA

    ACT["Action<br/>发起 Tool Call"]
    ACT --> OBS["Observation<br/>Tool Message 写回"]
    OBS -->|"回到 Thought"| TH

    FA(["Final Answer<br/>31°C · Mostly clear · 湿度 75%"])

    classDef thought fill:#a78bfa,stroke:#7c3aed,stroke-width:2px,color:#2e1065
    classDef action fill:#38bdf8,stroke:#0284c7,stroke-width:2px,color:#0c4a6e
    classDef obs fill:#f472b6,stroke:#db2777,stroke-width:2px,color:#500724
    classDef answer fill:#4ade80,stroke:#16a34a,stroke-width:2px,color:#14532d
    classDef ask fill:#14b8a6,stroke:#0f766e,stroke-width:2px,color:#042f2e
    classDef decide fill:#fbbf24,stroke:#d97706,stroke-width:2px,color:#422006

    class Q ask
    class TH thought
    class DEC decide
    class ACT action
    class OBS obs
    class FA answer
```

### 环上每一圈的载荷（同一循环，不同内容）

| 圈 | Thought | Action | Observation | 决策 |
|---|---|---|---|---|
| ↻1 | 优先用专用天气工具 | `get_weather_data("Shanghai")` | ❌ fetch 失败 | 不够 → 继续环 |
| ↻2 | 天气失败，改搜索 | `tavily_search(current weather…)` | ⚠️ 偏八月预报 | 不够 → 继续环 |
| ↻3 | refine 成 today/now | `tavily_search(today now…)` | ✅ 88°F / 75% | **够了 → 出环** |

**对照日志：** `AIMessage(tool_calls=...)` = Action，`ToolMessage` = Observation，中间自然语言（如 “Let me try looking it up another way”）= Thought。